In [45]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from collections import Counter

In [46]:
dataset_csv = r'C:\Cursos_Rebelway\ML_for_3D_and_VFX_MAY2025\myDataSets\nlp_emotions\combined_emotion.csv'
df = pd.read_csv(dataset_csv)

In [47]:
df.head()


,sentence,emotion
0,i just feel really helpless and heavy hearted,fear
1,ive enjoyed being able to slouch about relax a...,sad
2,i gave up my internship with the dmrg and am f...,fear
3,i dont know i feel so lost,sad
4,i am a kindergarten teacher and i am thoroughl...,fear


In [48]:
df.describe()

,sentence,emotion
count,422746,422746
unique,393822,6
top,i feel more adventurous willing to take risks ...,joy
freq,16,143067


In [49]:
texts = df['sentence'].tolist()
labels = df['emotion'].tolist()


In [50]:
def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {word: idx + 2 for idx, (word, freq) in enumerate(counter.items()) if freq >= min_freq}
    vocab['<PAD>'] = len(vocab)
    vocab['<UNK>'] = len(vocab)
    print(f"vocabulary: \n-----\n{vocab}\n-----")
    print(f"Vocabulary size: {len(vocab)}")
    return vocab

In [51]:
vocab = build_vocab(texts)
vocab

vocabulary: 
-----
{'i': 2, 'just': 3, 'feel': 4, 'really': 5, 'helpless': 6, 'and': 7, 'heavy': 8, 'hearted': 9, 'ive': 10, 'enjoyed': 11, 'being': 12, 'able': 13, 'to': 14, 'slouch': 15, 'about': 16, 'relax': 17, 'unwind': 18, 'frankly': 19, 'needed': 20, 'it': 21, 'after': 22, 'those': 23, 'last': 24, 'few': 25, 'weeks': 26, 'around': 27, 'the': 28, 'end': 29, 'of': 30, 'uni': 31, 'expo': 32, 'have': 33, 'lately': 34, 'started': 35, 'find': 36, 'myself': 37, 'feeling': 38, 'a': 39, 'bit': 40, 'listless': 41, 'which': 42, 'is': 43, 'never': 44, 'good': 45, 'thing': 46, 'gave': 47, 'up': 48, 'my': 49, 'internship': 50, 'with': 51, 'dmrg': 52, 'am': 53, 'distraught': 54, 'dont': 55, 'know': 56, 'so': 57, 'lost': 58, 'kindergarten': 59, 'teacher': 60, 'thoroughly': 61, 'weary': 62, 'job': 63, 'having': 64, 'taken': 65, 'university': 66, 'entrance': 67, 'exam': 68, 'suffered': 69, 'from': 70, 'anxiety': 71, 'for': 72, 'as': 73, 'did': 74, 'not': 75, 'want': 76, 'carry': 77, 'on': 78, 'wo

{'i': 2,
 'just': 3,
 'feel': 4,
 'really': 5,
 'helpless': 6,
 'and': 7,
 'heavy': 8,
 'hearted': 9,
 'ive': 10,
 'enjoyed': 11,
 'being': 12,
 'able': 13,
 'to': 14,
 'slouch': 15,
 'about': 16,
 'relax': 17,
 'unwind': 18,
 'frankly': 19,
 'needed': 20,
 'it': 21,
 'after': 22,
 'those': 23,
 'last': 24,
 'few': 25,
 'weeks': 26,
 'around': 27,
 'the': 28,
 'end': 29,
 'of': 30,
 'uni': 31,
 'expo': 32,
 'have': 33,
 'lately': 34,
 'started': 35,
 'find': 36,
 'myself': 37,
 'feeling': 38,
 'a': 39,
 'bit': 40,
 'listless': 41,
 'which': 42,
 'is': 43,
 'never': 44,
 'good': 45,
 'thing': 46,
 'gave': 47,
 'up': 48,
 'my': 49,
 'internship': 50,
 'with': 51,
 'dmrg': 52,
 'am': 53,
 'distraught': 54,
 'dont': 55,
 'know': 56,
 'so': 57,
 'lost': 58,
 'kindergarten': 59,
 'teacher': 60,
 'thoroughly': 61,
 'weary': 62,
 'job': 63,
 'having': 64,
 'taken': 65,
 'university': 66,
 'entrance': 67,
 'exam': 68,
 'suffered': 69,
 'from': 70,
 'anxiety': 71,
 'for': 72,
 'as': 73,
 'did': 

In [52]:
# vectorize the texts
def vectorize_texts(text, vocab):
    words = text.lower().split()
    vectorized_text = [vocab.get(word, vocab['<UNK>']) for word in words]
    return vectorized_text

vectorized_texts = [vectorize_texts(text, vocab) for text in texts]

labels_to_index = {label: idx for idx, label in enumerate(set(labels))}
numerical_labels = [labels_to_index[label] for label in labels]

In [54]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]
    

In [55]:
# pad seq
def collate_fn(batch):
    texts, labels = zip(*batch)
    max_length = max(len(text) for text in texts)
    padded_texts = [text + [vocab['<PAD>']] * (max_length - len(text)) for text in texts]
    return torch.tensor(padded_texts, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

In [56]:
X_train, X_test, y_train, y_test = train_test_split(vectorized_texts, numerical_labels, test_size=0.2, random_state=42)

train_dataset = EmotionDataset(X_train, y_train)
test_dataset = EmotionDataset(X_test, y_test)   

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [62]:
# model
class BilSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(BilSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab['<PAD>'])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim *2 , output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        out = self.fc(lstm_out[:, -1, :])
        return out

In [63]:
# init model
vocab_size = len(vocab) 
embedding_dim = 100
hidden_dim = 64 
output_dim = len(labels_to_index)


In [64]:
model = BilSTM(vocab_size, embedding_dim, hidden_dim, output_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [65]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
criterion.to(device)

CrossEntropyLoss()

In [72]:
epochs = 20 

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for texts, labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# evaluate the model
model.eval()
correct_predictions = 0
total_predictions = 0
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        predictions = model(texts)
        _, predicted_labels = torch.max(predictions, 1)

        correct_predictions += (predicted_labels == labels).sum().item()
        total_predictions += labels.size(0)

accuracy = correct_predictions / total_predictions

print(f"Test Accuracy: {accuracy:.4f}")


Epoch [1/20], Loss: 0.0764
Epoch [2/20], Loss: 0.0762
Epoch [3/20], Loss: 0.0757
Epoch [4/20], Loss: 0.0756
Epoch [5/20], Loss: 0.0753
Epoch [6/20], Loss: 0.0750
Epoch [7/20], Loss: 0.0750
Epoch [8/20], Loss: 0.0745
Epoch [9/20], Loss: 0.0744
Epoch [10/20], Loss: 0.0747
Epoch [11/20], Loss: 0.0746
Epoch [12/20], Loss: 0.0740
Epoch [13/20], Loss: 0.0739
Epoch [14/20], Loss: 0.0736
Epoch [15/20], Loss: 0.0736
Epoch [16/20], Loss: 0.0735
Epoch [17/20], Loss: 0.0730
Epoch [18/20], Loss: 0.0733
Epoch [19/20], Loss: 0.0730
Epoch [20/20], Loss: 0.0732
Test Accuracy: 0.9149


In [79]:
def predict_emotion(text, model, vocab, labels_to_index, device):
    model.eval()
    with torch.no_grad():
        vectorized_text = vectorize_texts(text, vocab)
        if not vectorized_text:  # Handle empty text case
            return "unknown"
        
    padded_text = [vectorized_text + [vocab['<PAD>']] * (max(len(vectorized_text),1) - len(vectorized_text))]
    input_tensor = torch.tensor(padded_text, dtype=torch.long).to(device)
    prediction = model(input_tensor)
    _, predicted_label = torch.max(prediction, 1)

    index_to_labels = {idx: label for label, idx in labels_to_index.items()}
    return index_to_labels[predicted_label.item()]

new_sentences = ["I'm so happy today!", "This is terrifying.", "I feel angry about the situation.", "It's a neutral day for me.", "it's raining outside, I feel calm."]
for sentence in new_sentences:
    emotion = predict_emotion(sentence, model, vocab, labels_to_index, device)
    print(f"Sentence: '{sentence}' - Predicted Emotion: {emotion}")

Sentence: 'I'm so happy today!' - Predicted Emotion: joy
Sentence: 'This is terrifying.' - Predicted Emotion: joy
Sentence: 'I feel angry about the situation.' - Predicted Emotion: anger
Sentence: 'It's a neutral day for me.' - Predicted Emotion: joy
Sentence: 'it's raining outside, I feel calm.' - Predicted Emotion: joy
